In [14]:
import pandas as pd
import numpy as np

df_trans = pd.read_csv('data/train_transaction.csv', low_memory=False)

print(df_trans.shape)
print(df_trans.dtypes.head(10))
print(df_trans.head(3))
print(df_trans['isFraud'].value_counts(normalize=True))


(590540, 394)
TransactionID       int64
isFraud             int64
TransactionDT       int64
TransactionAmt    float64
ProductCD          object
card1               int64
card2             float64
card3             float64
card4              object
card5             float64
dtype: object
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   

  V336  V337  V338  V339  
0  NaN   NaN   NaN   NaN  
1  NaN   NaN   NaN   NaN  
2  NaN   NaN 

In [6]:
df_id = pd.read_csv('data/train_identity.csv', low_memory=False)

print(f"Transactions : {df_trans.shape}")
print(f"Identity     : {df_id.shape}")


common = df_trans['TransactionID'].isin(df_id['TransactionID']).sum()
print(f"Transactions with identity : {common:,}")
print(f"Transaction without        : {len(df_trans) - common:,}")


Transactions : (590540, 394)
Identity     : (144233, 41)
Transactions with identity : 144,233
Transaction without        : 446,307


In [16]:
df_left = pd.merge(df_trans, df_id, on='TransactionID', how='left')

df_inner = pd.merge(df_trans, df_id, on='TransactionID', how='inner')

print(f"LEFT shape : {df_left.shape}")
print(f"INNER shape : {df_inner.shape}")

print(f"Rows lost with INNER : {len(df_left) - len(df_inner):,}")

print(f"Fraud rate LEFT : {df_left['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate INNER : {df_inner['isFraud'].mean()*100:.2f}%")

LEFT shape : (590540, 434)
INNER shape : (144233, 434)
Rows lost with INNER : 446,307
Fraud rate LEFT : 3.50%
Fraud rate INNER : 7.85%


In [11]:
df = df_left.copy()

missing = df.isnull().mean().sort_values(ascending=False)

print("Top 10 empty columns:")
print(missing.head(10))

for threshold in [0.10, 0.30, 0.50, 0.80, 0.90]:
    n_dropped = (missing > threshold).sum()
    n_kept = (missing <= threshold).sum()
    print(f"Threshold {threshold:.0%} -> drop {n_dropped} cols, keep {n_kept} cols")

Top 10 empty columns:
id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
dtype: float64
Threshold 10% -> drop 322 cols, keep 112 cols
Threshold 30% -> drop 232 cols, keep 202 cols
Threshold 50% -> drop 214 cols, keep 220 cols
Threshold 80% -> drop 74 cols, keep 360 cols
Threshold 90% -> drop 12 cols, keep 422 cols


In [12]:
counts = df['isFraud'].value_counts()
pct    = df['isFraud'].value_counts(normalize=True) * 100

print(f"Legit (0) : {counts[0]:>8,}  ({pct[0]:.1f}%)")
print(f"Fraud (1) : {counts[1]:>8,}  ({pct[1]:.1f}%)")


dummy_acc = counts[0] / len(df)
print(f"\nDummy accuracy (always legit) : {dummy_acc*100:.2f}%")
print(f"Fraud it catches              : 0")
print(f"This is your baseline. Any model must beat it meaningfully.")

Legit (0) :  569,877  (96.5%)
Fraud (1) :   20,663  (3.5%)

Dummy accuracy (always legit) : 96.50%
Fraud it catches              : 0
This is your baseline. Any model must beat it meaningfully.
